# Нефтекод 2026 — EDA

Разведочный анализ данных задачи **прогнозирования окислительной стабильности смазочных материалов**: двухвыходная регрессия на уровне сценария (рецептура + условия Daimler Oxidation Test).

**Целевые:**
- `Delta Kin. Viscosity KV100 (relative), %` — относительное изменение кинематической вязкости после DOT
- `Oxidation EOT, A/cm` — ИК-поглощение окисления (DIN 51453)

**Входы:**
- `daimler_mixtures_train.csv` / `daimler_mixtures_test.csv` — длинные таблицы: одна строка = один компонент одного сценария + условия теста (дублирующиеся по компонентам)
- `daimler_component_properties.csv` — длинная таблица свойств компонента×партии

Ноутбук запускается из корня репозитория (`jupyter lab`/`jupyter notebook` в `neftecode-2026/`).

## 0. Импорт и загрузка

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up until we find data/raw — работает и из notebooks/, и из корня."""
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("data/raw не найден в родительских директориях")


REPO = find_repo_root()
RAW = REPO / "data" / "raw"
print("repo root:", REPO)
print("raw files:", sorted(p.name for p in RAW.glob("*.csv")))

In [ ]:
# Оригинальные заголовки в данных — длинные русские строки. Приводим к коротким
# техническим именам: это облегчает весь дальнейший анализ и моделирование.

MIX_RENAME = {
    "Компонент": "component_id",
    "Наименование партии": "batch_id",
    "Массовая доля, %": "share_pct",
    "Температура испытания | ASTM D445 Daimler Oxidation Test (DOT), °C": "T_C",
    "Время испытания | - Daimler Oxidation Test (DOT), ч": "t_h",
    "Количество биотоплива | - Daimler Oxidation Test (DOT), % масс": "biofuel_pct",
    "Дозировка катализатора, категория": "catalyst_cat",
    "Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %": "target_dkv",
    "Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm": "target_eot",
}

PROP_RENAME = {
    "Компонент": "component_id",
    "Наименование партии": "batch_id",
    "Наименование показателя": "property_name",
    "Единица измерения_по_партиям": "unit",
    "Значение показателя": "value_raw",
}

# encoding='utf-8-sig' нужен: в файлах есть BOM перед scenario_id
train = pd.read_csv(RAW / "daimler_mixtures_train.csv", encoding="utf-8-sig").rename(columns=MIX_RENAME)
test = pd.read_csv(RAW / "daimler_mixtures_test.csv", encoding="utf-8-sig").rename(columns=MIX_RENAME)
props = pd.read_csv(RAW / "daimler_component_properties.csv", encoding="utf-8-sig").rename(columns=PROP_RENAME)

# batch_id нужно хранить как строку — бывают значения '32', '01050.22', 'typical'
for df in (train, test, props):
    df["batch_id"] = df["batch_id"].astype(str).str.strip()

print("train:", train.shape, "| test:", test.shape, "| props:", props.shape)
train.head(3)

## 1. Структура рецептур (mixtures)

- Сколько сценариев в train/test;
- Сколько компонентов в каждом сценарии;
- Распределение массовых долей;
- Диапазоны условий DOT (T, t, biofuel, catalyst).

In [ ]:
n_train_scen = train["scenario_id"].nunique()
n_test_scen = test["scenario_id"].nunique()
print(f"Сценариев train: {n_train_scen}")
print(f"Сценариев test:  {n_test_scen}")
print(f"Строк train: {len(train)} (в среднем {len(train) / n_train_scen:.2f} компонентов/сценарий)")
print(f"Строк test:  {len(test)} (в среднем {len(test) / n_test_scen:.2f} компонентов/сценарий)")

In [ ]:
# Число компонентов на сценарий

comps_per_scen_train = train.groupby("scenario_id").size()
comps_per_scen_test = test.groupby("scenario_id").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.histplot(comps_per_scen_train, bins=range(1, int(comps_per_scen_train.max()) + 2), ax=axes[0])
axes[0].set_title(f"train: компонентов в сценарии (mean={comps_per_scen_train.mean():.1f})")
axes[0].set_xlabel("число компонентов")
sns.histplot(comps_per_scen_test, bins=range(1, int(comps_per_scen_test.max()) + 2), ax=axes[1], color="C1")
axes[1].set_title(f"test: компонентов в сценарии (mean={comps_per_scen_test.mean():.1f})")
axes[1].set_xlabel("число компонентов")
plt.tight_layout();
plt.show()

print("train describe:\n", comps_per_scen_train.describe().round(2).to_string())
print("\ntest describe:\n", comps_per_scen_test.describe().round(2).to_string())

In [ ]:
# Распределение массовых долей. Проверяем также, суммируются ли доли в ~100% на сценарий.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train["share_pct"], bins=40, ax=axes[0])
axes[0].set_title("train: распределение массовых долей компонентов")
axes[0].set_xlabel("share, %")

share_sum_train = train.groupby("scenario_id")["share_pct"].sum()
share_sum_test = test.groupby("scenario_id")["share_pct"].sum()
sns.histplot(share_sum_train, bins=40, ax=axes[1], label="train")
sns.histplot(share_sum_test, bins=40, ax=axes[1], label="test", color="C1", alpha=0.6)
axes[1].axvline(100, color="red", ls="--", lw=1)
axes[1].set_title("Сумма долей на сценарий (ожидаем ~100%)")
axes[1].set_xlabel("sum(share), %")
axes[1].legend()
plt.tight_layout();
plt.show()

print("share per row — train describe:\n", train["share_pct"].describe().round(3).to_string())
print("\nsum of shares per scenario — train describe:\n", share_sum_train.describe().round(3).to_string())
print("\nsum of shares per scenario — test describe:\n", share_sum_test.describe().round(3).to_string())

In [ ]:
# Условия DOT. Берём одну строку на сценарий (условия одинаковые для всех его компонентов).

COND_COLS = ["T_C", "t_h", "biofuel_pct", "catalyst_cat"]
cond_train = train.groupby("scenario_id")[COND_COLS].first()
cond_test = test.groupby("scenario_id")[COND_COLS].first()

print("train — условия DOT:")
print(cond_train.describe().round(2).to_string())
print("\ntest — условия DOT:")
print(cond_test.describe().round(2).to_string())

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(COND_COLS):
    sns.histplot(cond_train[col], ax=axes[0, i], bins=20)
    axes[0, i].set_title(f"train: {col}")
    sns.histplot(cond_test[col], ax=axes[1, i], bins=20, color="C1")
    axes[1, i].set_title(f"test:  {col}")
plt.tight_layout();
plt.show()

## 2. Таксономия компонентов

Имя компонента имеет вид `<класс>_<индекс>` (например `Антиоксидант_5`, `Противоизносная_присадка_20`, `Базовое_масло_14`). Префикс — всё кроме финального `_<число>` — и есть класс присадки. Смотрим, сколько компонентов и партий в каждом классе и как часто они встречаются в рецептурах train.

In [ ]:
PREFIX_RE = re.compile(r"_(\d+)$")


def extract_prefix(comp: str) -> str:
    """Убираем финальный `_<число>` (`Антиоксидант_5` -> `Антиоксидант`)."""
    return PREFIX_RE.sub("", comp)


all_components = (
    pd.concat(
        [
            train[["component_id", "batch_id"]],
            test[["component_id", "batch_id"]],
            props[["component_id", "batch_id"]],
        ],
        ignore_index=True,
    )
    .drop_duplicates()
)
all_components["prefix"] = all_components["component_id"].apply(extract_prefix)

# Частота использования в train-рецептурах (на уровне сценариев)
train_with_prefix = train.assign(prefix=train["component_id"].apply(extract_prefix))
freq_in_scen = (
    train_with_prefix.groupby("prefix")["scenario_id"].nunique() / train["scenario_id"].nunique()
)

taxonomy = (
    all_components.groupby("prefix")
    .agg(n_components=("component_id", "nunique"), n_batches=("batch_id", "nunique"))
    .assign(share_of_train_scenarios=freq_in_scen)
    .fillna({"share_of_train_scenarios": 0.0})
    .sort_values("n_components", ascending=False)
)
taxonomy["share_of_train_scenarios"] = taxonomy["share_of_train_scenarios"].round(3)
taxonomy

In [ ]:
fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(taxonomy))))
taxonomy["n_components"].plot.barh(ax=ax, color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("# уникальных компонентов")
ax.set_title("Таксономия компонентов по префиксу")
plt.tight_layout();
plt.show()

## 3. Целевые переменные

Цели заданы на уровне сценария — они продублированы по всем строкам компонентов одного сценария. Берём по одному значению на сценарий и смотрим:
- маргинальные распределения;
- совместное распределение;
- корреляции с условиями DOT;
- знак `Delta KV100`: положительные значения — полимеризация/загустевание, отрицательные — крекинг/разжижение. Баланс важен для понимания физики и выбора loss.

In [ ]:
targets = (
    train.groupby("scenario_id")[["target_dkv", "target_eot", *COND_COLS]].first().reset_index()
)
print("цели на сценарий:")
print(targets[["target_dkv", "target_eot"]].describe().round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(targets["target_dkv"], bins=40, ax=axes[0])
axes[0].axvline(0, color="red", ls="--", lw=1)
axes[0].set_title("Delta KV100 (relative), %")

sns.histplot(targets["target_eot"], bins=40, ax=axes[1], color="C2")
axes[1].set_title("Oxidation EOT, A/cm")

sns.scatterplot(data=targets, x="target_dkv", y="target_eot", ax=axes[2], alpha=0.6)
axes[2].axvline(0, color="red", ls="--", lw=1)
axes[2].set_title("совместное: EOT vs DKV")
plt.tight_layout();
plt.show()

In [ ]:
# Баланс знаков Delta KV100 — крекинг vs загустевание

sign_counts = pd.Series(
    {
        "negative (< 0, крекинг/разжижение)": int((targets["target_dkv"] < 0).sum()),
        "zero (== 0)": int((targets["target_dkv"] == 0).sum()),
        "positive (> 0, полимеризация)": int((targets["target_dkv"] > 0).sum()),
    }
)
sign_counts_frac = (sign_counts / sign_counts.sum()).round(3)
print("Баланс знаков Delta KV100:")
print(pd.concat([sign_counts.rename("n"), sign_counts_frac.rename("share")], axis=1).to_string())

In [ ]:
# Корреляция целей с условиями DOT

corr = targets[["target_dkv", "target_eot", *COND_COLS]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Spearman: цели ↔ условия DOT")
plt.tight_layout();
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(COND_COLS):
    sns.stripplot(data=targets, x=col, y="target_dkv", ax=axes[0, i], jitter=0.25, alpha=0.6)
    axes[0, i].axhline(0, color="red", ls="--", lw=0.8)
    axes[0, i].set_title(f"Delta KV100 vs {col}")
    sns.stripplot(data=targets, x=col, y="target_eot", ax=axes[1, i], jitter=0.25, alpha=0.6, color="C2")
    axes[1, i].set_title(f"EOT vs {col}")
plt.tight_layout();
plt.show()

## 4. Свойства компонентов

`daimler_component_properties.csv` — длинная таблица (component × batch × property). Разбираемся:
- сколько уникальных `property_name` и какие единицы у них;
- покрытие: для какой доли встречающихся в данных пар `(component, batch)` есть хоть какое-то свойство;
- топ-20 самых частых свойств и доля пропусков у каждого по множеству всех пар.

In [ ]:
print("уникальных property_name:", props["property_name"].nunique())
print("уникальных (property_name, unit):", props.groupby(["property_name", "unit"]).ngroups)
print("\nтоп-10 единиц измерения:")
print(props["unit"].value_counts().head(10).to_string())

In [ ]:
# Универсум пар (component, batch) из train+test — для них нужны свойства в модели.

mix_pairs = (
    pd.concat([train[["component_id", "batch_id"]], test[["component_id", "batch_id"]]])
    .drop_duplicates()
    .reset_index(drop=True)
)
print("пар (component, batch) в train+test:", len(mix_pairs))

prop_pairs = props[["component_id", "batch_id"]].drop_duplicates()
prop_pairs_set = set(map(tuple, prop_pairs.to_numpy()))

mix_pairs["has_any_prop"] = [
    (c, b) in prop_pairs_set for c, b in mix_pairs[["component_id", "batch_id"]].to_numpy()
]
# Если партии нет — возможно, есть 'typical'-строка на тот же component_id
typical_comps = set(props.loc[props["batch_id"].str.lower() == "typical", "component_id"])
mix_pairs["has_typical"] = mix_pairs["component_id"].isin(typical_comps)
mix_pairs["covered"] = mix_pairs["has_any_prop"] | mix_pairs["has_typical"]

print(
    f"покрытие прямое (та же партия):       "
    f"{mix_pairs['has_any_prop'].mean():.1%}"
)
print(
    f"покрытие с учётом typical-фолбэка:    "
    f"{mix_pairs['covered'].mean():.1%}"
)

In [ ]:
# Топ-20 свойств: по числу уникальных (component, batch), для которых они заданы

pairs_per_prop = (
    props[["property_name", "component_id", "batch_id"]]
    .drop_duplicates()
    .groupby("property_name")
    .size()
    .rename("n_pairs")
)
extra = props.groupby("property_name").agg(
    n_components=("component_id", "nunique"), n_units=("unit", "nunique")
)
top_props = (
    pd.concat([pairs_per_prop, extra], axis=1)
    .sort_values("n_pairs", ascending=False)
    .head(20)
)
top_props["share_of_mix_pairs"] = (top_props["n_pairs"] / len(mix_pairs)).round(3)
top_props["missing_share_of_mix_pairs"] = (1 - top_props["share_of_mix_pairs"]).round(3)
top_props

In [ ]:
# Визуализация покрытия топ-20 свойств

fig, ax = plt.subplots(figsize=(10, 7))
top_props["share_of_mix_pairs"].iloc[::-1].plot.barh(ax=ax, color="teal")
ax.set_xlim(0, 1)
ax.set_xlabel("доля пар (component, batch) из train+test, где свойство известно")
ax.set_title("Покрытие топ-20 свойств")
plt.tight_layout();
plt.show()

## 5. Typical vs измеренные значения

Часть строк в `component_properties.csv` имеет `batch_id == 'typical'` — это "паспортные" значения производителя. Проверяем, сколько компонентов одновременно имеют и измеренные (по конкретной партии), и `typical` значения, и насколько они расходятся — это подскажет, можно ли смело использовать `typical` как фолбэк при отсутствии партийных данных.

In [ ]:
def to_float(x):
    """Пытаемся распарсить значение. В данных встречаются запятые, пробелы, диапазоны."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", ".").replace(" ", "")
    # диапазоны вида '10-20' -> середина
    m = re.fullmatch(r"(-?\d+(?:\.\d+)?)-(-?\d+(?:\.\d+)?)", s)
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    s = re.sub(r"^[<>≤≥]=?", "", s)
    try:
        return float(s)
    except ValueError:
        return np.nan


props = props.assign(value=props["value_raw"].apply(to_float))
props["is_typical"] = props["batch_id"].str.lower() == "typical"
print("строк всего:", len(props))
print("typical строк:", int(props["is_typical"].sum()))
print("числовых значений:", int(props["value"].notna().sum()))

In [ ]:
# Для каждого свойства: для скольких component_id есть и measured и typical;
# считаем относительную разницу между "среднемеренным" и typical.

typical_vals = (
    props[props["is_typical"]]
    .groupby(["component_id", "property_name"])["value"]
    .mean()
    .rename("typical")
)
measured_vals = (
    props[~props["is_typical"]]
    .groupby(["component_id", "property_name"])["value"]
    .mean()
    .rename("measured_mean")
)

compare = pd.concat([typical_vals, measured_vals], axis=1).dropna()
compare["abs_rel_diff"] = (compare["measured_mean"] - compare["typical"]).abs() / (
    compare["typical"].abs().clip(lower=1e-9)
)
print(f"компонент×свойство пар с обоими значениями: {len(compare)}")
print(f"уникальных компонентов в сравнении:         {compare.index.get_level_values(0).nunique()}")
print("\nраспределение |measured - typical| / |typical|:")
print(compare["abs_rel_diff"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(3).to_string())

In [ ]:
# Какие свойства больше всего "гуляют" между typical и measured?

rel_by_prop = (
    compare.groupby("property_name")["abs_rel_diff"]
    .agg(["count", "median", "mean", "max"])
    .sort_values("median", ascending=False)
    .head(15)
    .round(3)
)
rel_by_prop

## 6. Перекрытие train/test

Узнаём, что именно тестируется — обобщение по рецептурам при известных партиях или холодный старт по новым компонентам/партиям. Это определяет выбор признаков и стратегии валидации.

In [ ]:
train_comps = set(train["component_id"])
test_comps = set(test["component_id"])
train_batches = set(map(tuple, train[["component_id", "batch_id"]].drop_duplicates().to_numpy()))
test_batches = set(map(tuple, test[["component_id", "batch_id"]].drop_duplicates().to_numpy()))

new_comps = test_comps - train_comps
new_pairs = test_batches - train_batches
# Новые партии известного компонента — в test есть (c,b), (c,*) нет в train, но c есть в train
new_batches_known_comp = {
    (c, b) for (c, b) in new_pairs if c in train_comps
}

print(f"компонентов в train: {len(train_comps)} | в test: {len(test_comps)}")
print(f"  компонентов пересечения: {len(train_comps & test_comps)}")
print(f"  новых компонентов (только test): {len(new_comps)}")
print(f"\nпар (component,batch) в train: {len(train_batches)} | в test: {len(test_batches)}")
print(f"  пар пересечения: {len(train_batches & test_batches)}")
print(f"  новых пар:                     {len(new_pairs)}")
print(f"    из них — новая партия известного компонента: {len(new_batches_known_comp)}")
print(f"    из них — совсем новый компонент:             {len(new_pairs) - len(new_batches_known_comp)}")
if new_comps:
    print("\nновые компоненты в test:")
    print(sorted(new_comps))

In [ ]:
# Доля компонентов test-рецептур, для которых в train-рецептурах вообще
# встречался тот же component_id (не обязательно та же партия).

test["component_in_train"] = test["component_id"].isin(train_comps)
cov_by_scen = test.groupby("scenario_id")["component_in_train"].mean()
print("доля компонентов test-сценария, встречавшихся в train:")
print(cov_by_scen.describe().round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(cov_by_scen, bins=20, ax=ax)
ax.set_xlabel("доля компонентов test-сценария, известных из train")
ax.set_title("Покрытие test-сценариев по составу")
plt.tight_layout();
plt.show()

## 7. Сводка по условиям DOT

Сколько уникальных комбинаций `(T, t, catalyst)` в train и test, и сколько сценариев приходится на каждую.

In [ ]:
cond_key_cols = ["T_C", "t_h", "catalyst_cat"]
cond_train_grp = cond_train.groupby(cond_key_cols).size().rename("n_train_scen")
cond_test_grp = cond_test.groupby(cond_key_cols).size().rename("n_test_scen")
cond_summary = (
    pd.concat([cond_train_grp, cond_test_grp], axis=1)
    .fillna(0)
    .astype(int)
    .sort_values("n_train_scen", ascending=False)
)
print(f"уникальных комбинаций (T, t, catalyst) — train: {cond_train_grp.shape[0]}, test: {cond_test_grp.shape[0]}")
print(f"комбинаций test, отсутствующих в train: {(cond_summary['n_train_scen'] == 0).sum()}")
cond_summary

In [ ]:
# То же с biofuel — отдельно, т.к. именно он обычно задаёт "жёсткость" DOT
cond_full = cond_train.groupby(["T_C", "t_h", "biofuel_pct", "catalyst_cat"]).size()
print(f"train: {len(cond_full)} уникальных комбинаций (T, t, biofuel, catalyst)")
print(cond_full.sort_values(ascending=False).head(15).to_string())

---

## Выводы для моделирования

1. **Задача — двухвыходная регрессия на уровне сценария.** Исходные таблицы длинные (строка = компонент), поэтому до моделирования нужно агрегировать рецептуру в один вектор признаков на `scenario_id`; цели и условия DOT при этом дублируются — это нормально и надо сворачивать через `groupby('scenario_id').first()`.
2. **Знаки `Delta KV100` неоднородны** — есть и положительные (полимеризация/загустевание), и отрицательные (крекинг/разжижение) сценарии. Значит, модель должна уметь предсказывать знак, а в лоссе лучше держать RMSE/Huber по "сырому" значению, а не по `log|Delta|`. При желании можно добавить вспомогательную классификационную голову на знак.
3. **Ключевой признаковый контур — взвешенные по доле свойства компонентов.** Свойства лежат в длинной таблице по парам `(component, batch)`, поэтому пайплайн: (а) распарсить `value` в числа (есть `,` вместо `.`, диапазоны `a-b`, символы `<>`); (б) развернуть в таблицу `pair → property → value`; (в) для сценария взять свёртку `∑ share_i × value_i` (+ `max`, `min`, доля-взвешенная дисперсия) по каждому свойству и каждому префиксу класса.
4. **Покрытие свойствами неполное.** Для части пар `(component, batch)` партийные значения отсутствуют — но есть строки `batch_id == 'typical'`. Нужен двухуровневый фолбэк: сначала измерения по своей партии, затем `typical` того же компонента, затем среднее по префиксу класса. Разрыв `typical` vs измеренные по топ-расходящимся свойствам (см. § 5) стоит заложить в кросс-валидацию — проверить, не ухудшает ли фолбэк модель систематически.
5. **Условия DOT нужно включать как признаки явно и нелинейно.** `T`, `t`, `biofuel_pct`, `catalyst_cat` дают видимые монотонные сдвиги обеих целей, но комбинаций мало (см. § 7). Для градиентного бустинга это ок "как есть"; для линейной регрессии нужны взаимодействия `condition × свойство` (особенно `biofuel × содержание антиоксиданта/серы`).
6. **Стратегия валидации — GroupKFold по `scenario_id`**, а не обычный K-Fold: в train нет повторяющихся сценариев, но строки одного сценария дублируются. Для оценки обобщения дополнительно имитируем test-режим: отдельный фолд, где для валидации выбираются сценарии с *новой парой* `(component, batch)` (если в test есть новые партии или компоненты — см. § 6) — это покажет, работает ли фолбэк из п. 4.
7. **Таксономия компонентов по префиксу (§ 2) — ценный категориальный сигнал.** Можно строить агрегаты "доля × свойство" отдельно по каждому классу (базовое масло, антиоксидант, дисперсант, детергент, противоизносная присадка, модификатор трения, соединение молибдена, …), а также считать балансы: суммарная доля каждого класса в рецептуре и соотношения (напр. `share(антиоксидант) / share(базовое масло)`) — это прямые физико-химические ручки окислительной стабильности.